# PT-RL-SYSTEM. Rollout, Training and Reward System | 采样、训练与奖励系统

**难度：** Hard | **环境：** CPU-first | **标签：** `后训练`, `强化学习系统`, `资源调度`

---

## 本节导读

在线后训练把轨迹生成、参考策略计算、奖励或验证、参数更新和权重分发连接成循环。单个角色跑得快不代表系统稳定：候选展开比例、批量聚合、队列增长、状态驻留和同步失败都会改变端到端容量。

系统比较先把轨迹生成、奖励和学习器的处理率换算成相同的工作单位，再检查各资源池的状态驻留、共享链路占用、版本确认和恢复条件。账本通过表示候选可以进入真实多工作进程实验，不表示已经获得吞吐收益。

**关键词：** 轨迹生成器、学习器、共置、分离部署、流水线瓶颈

![后训练系统角色与数据流](../docs/public/02_PyTorch_Algorithms/pt_rl_system_topology.svg)

## 前置阅读

**导语：** 先理解轨迹身份和策略同步，再把算法角色放入多阶段系统。

- [Part 02 · PT-ROLLOUT-LIFECYCLE Rollout 与策略新鲜度](./PT-ROLLOUT-LIFECYCLE_Rollout_and_Policy_Freshness.ipynb)
- [Part 02 · 16 PPO 资源与状态](./16_RLHF_PPO_Memory.ipynb)

### Step 1：先固定角色、状态和阶段倍率

角色可以共享 GPU，也可以拆到不同资源池。选择拓扑前，要同时明确每个角色持有的状态、输入输出和阶段倍率。例如一个提示可能生成 `G` 条候选轨迹，奖励阶段处理 `G` 个回答，而学习器等待若干提示及其候选聚合成一次更新；这些对象不能都笼统记作一个“样本”。

因此系统账本先选定统一工作单位，例如“每个提示组”或“每条轨迹”，再把各阶段处理率换算到同一单位。例如每个提示生成 4 条轨迹时，奖励器每秒处理 80 条轨迹，只相当于每秒处理 20 个提示组；若提示组到达率为每秒 24 组，奖励队列仍会持续增长。角色持有的状态和阶段倍率共同决定容量与队列。

| 角色 | 主要驻留状态 | 输入与输出 | 数量关系 |
| --- | --- | --- | --- |
| 轨迹生成器 | 当前策略、KV Cache、采样状态 | 提示 → 回答、行为对数概率与策略版本 | 每个提示生成 `G` 条候选 |
| 奖励 / 验证器 | 奖励模型或规则版本 | 轨迹 → 奖励、验证状态 | 通常逐轨迹处理 |
| 参考策略服务 | 冻结参考策略 | 轨迹 → 参考对数概率 | 按算法需要逐轨迹处理 |
| 学习器 | 策略、优化器、梯度与训练状态 | 合格批次 → 新检查点 | 聚合多个提示组后更新 |
| 产物协调器 | 版本、队列、哈希与恢复状态 | 新产物 → 分发确认与活动版本 | 每次更新触发一次同步 |

### Step 2：部署拓扑改变状态副本、资源争用和跨池边

共置减少跨池边，但会让策略、参考模型、奖励模型和优化器状态竞争同一资源池；分离允许角色独立扩缩容，却新增轨迹、对数概率和权重传输。混合拓扑只在明确哪些高频边需要共置、哪些重状态必须拆分后才有意义。

| 候选 | 状态驻留 | 新增传输与同步 | 故障与调度代价 | 首先验证 |
| --- | --- | --- | --- | --- |
| 共置 | 多类模型与训练状态共享资源池 | 跨池传输少，仍需阶段间状态交接 | 显存争用、阶段切换和故障域扩大 | 分阶段峰值显存、切换空转时间 |
| 分离 | 各角色独立驻留并可分别扩缩容 | 轨迹、对数概率、奖励和权重跨池 | 队列积压、版本滞后和跨池恢复 | 每条边的载荷率、确认延迟与积压 |
| 混合 | 高频交互角色共置，重状态拆分 | 只保留选定跨池边 | 配置复杂，局部争用与传输同时存在 | 状态副本、关键跨池边和失败切换 |

### Step 3：等价容量、传输预算和硬门槛共同决定准入

先依据阶段倍率，把采样、奖励和学习阶段的原始处理率换算为统一工作单位。例如以提示组为单位时，逐轨迹阶段的处理率需要除以每组候选数 `G`，学习阶段则按每次更新消费的提示组数换算。只有完成单位归一化后，最小等价处理率才能表示流水线容量；阶段等价处理率减到达率，才是可比较的队列余量。

跨池边需要把单位对象载荷乘以对象频率，再除以有效带宽，得到每秒传输占用。共享同一互连域（fabric）的边会争用带宽，应在域内聚合；不同互连域可以并行，不能把所有边简单相加后直接宣称为端到端关键路径时间。扣除可重叠比例后得到的仍是传输暴露预算代理。

拓扑准入分两层：显存超限、版本未确认或恢复路径缺失属于硬失败；队列增长或传输暴露过高说明候选需要调优。通过账本只表示候选值得进入真实多工作进程实验，不代表已经获得吞吐收益。

![候选拓扑的准入与瓶颈判断](../docs/public/02_PyTorch_Algorithms/pt_rl_system_decision.svg)

### Step 4：实现系统账本与候选准入

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：定位流水线瓶颈<br>`pipeline_capacity` | 将各阶段原始处理率换算为统一工作单位，再计算容量和队列余量 | 原始速率与每工作单位对象数必须为正 | 候选展开倍率、并列瓶颈、稳定与增长队列 |
| TODO 2：建立跨池传输账本<br>`transfer_ledger` | 将对象载荷、阶段倍率和频率换算为分链路传输占用 | 同池边不产生网络传输；独立链路不直接串行相加 | 同池、跨池、共享链路与独立链路 |
| TODO 3：执行拓扑准入<br>`evaluate_topology` | 联合队列、分池显存、暴露传输、版本和恢复门槛 | 硬门槛失败时拒绝，容量或传输不足时调优 | 可行、队列增长、分池超限、同步与恢复失败 |

In [ ]:
from typing import Dict, List


In [ ]:
def pipeline_capacity(stage_rates, arrival_rate, items_per_work_unit):
    """将原始阶段速率归一化后，返回容量、瓶颈和队列余量。"""
    # ==========================================
    # TODO 1：把对象处理率换算为统一工作单位每秒，再判断流水线稳定性。
    # effective_rates = ???  # 阶段原始速率除以每工作单位对象数。
    # queue_margins = ???  # 各阶段等价处理率减工作单位到达率。
    # ==========================================
    raise NotImplementedError("TODO 1：计算流水线容量与队列余量")

def transfer_ledger(edges, placement):
    """汇总跨资源池 payload；同池边不产生网络传输。"""
    # ==========================================
    # TODO 2：计算跨池边载荷，并按共享 fabric 聚合暴露传输占用。
    # traffic_bytes_per_s = ???  # 单对象字节数 × 每工作单位对象数 × 工作单位每秒。
    # critical_exposed_seconds_per_s = ???  # 各 fabric 暴露占用的最大值。
    # ==========================================
    raise NotImplementedError("TODO 2：建立跨池传输账本")

def evaluate_topology(capacity_report, memory_by_pool, memory_limits, transfer_report, version_sync_ok, recovery_ok, max_exposed_s_per_s=0.1):
    """返回 accept/tune/reject 及可复核原因。"""
    # ==========================================
    # TODO 3：依次检查分池显存、版本、恢复、队列和关键互连域暴露传输。
    # over_limit_pools = ???  # 驻留量超过预算的资源池。
    # decision = ???  # accept、tune 或 reject。
    # ==========================================
    raise NotImplementedError("TODO 3：执行系统拓扑准入")

### 测试

测试分别验证瓶颈归因、跨池载荷和系统准入。

In [ ]:
def test_capacity():
    """稳态容量必须由最慢阶段决定。"""
    result = pipeline_capacity(
        {'rollout': 240, 'reward': 160, 'learner': 80}, arrival_rate=70,
        items_per_work_unit={'rollout': 3, 'reward': 2, 'learner': 1},
    )
    assert result['effective_rates'] == {'rollout': 80.0, 'reward': 80.0, 'learner': 80.0}
    assert result['bottlenecks'] == ['learner', 'reward', 'rollout']
    assert result['queue_stable'] is True
    assert pipeline_capacity({'rollout': 120, 'reward': 80}, 90, {'rollout': 1, 'reward': 1})['queue_stable'] is False

def test_transfer():
    """只有跨资源池的边才能计入传输账本。"""
    edges = [
        {'source': 'rollout', 'target': 'reward', 'bytes_per_item': 100, 'items_per_work_unit': 2, 'work_units_per_second': 5, 'bandwidth_bytes_per_s': 1000, 'overlap_fraction': 0.5, 'fabric_id': 'pcie-0'},
        {'source': 'learner', 'target': 'rollout', 'bytes_per_item': 1000, 'items_per_work_unit': 1, 'work_units_per_second': 1, 'bandwidth_bytes_per_s': 2000, 'overlap_fraction': 0.0, 'fabric_id': 'pcie-1'},
    ]
    report = transfer_ledger(edges, {'rollout': 'A', 'reward': 'A', 'learner': 'B'})
    assert report['cross_pool_bytes_per_s'] == 1000
    assert report['fabric_exposed_seconds_per_s'] == {'pcie-1': 0.5}
    assert report['critical_exposed_seconds_per_s'] == 0.5

def test_gate():
    """容量不足返回 tune，显存、同步或恢复失败返回 reject。"""
    multipliers = {'rollout': 1, 'reward': 1}
    stable = pipeline_capacity({'rollout': 120, 'reward': 80}, 70, multipliers)
    unstable = pipeline_capacity({'rollout': 120, 'reward': 80}, 90, multipliers)
    low_transfer = {'critical_exposed_seconds_per_s': 0.05}
    assert evaluate_topology(stable, {'A': 30}, {'A': 40}, low_transfer, True, True)['decision'] == 'accept'
    assert evaluate_topology(unstable, {'A': 30}, {'A': 40}, low_transfer, True, True)['decision'] == 'tune'
    assert evaluate_topology(stable, {'A': 50}, {'A': 40}, low_transfer, True, True)['decision'] == 'reject'
    assert evaluate_topology(stable, {'A': 30}, {'A': 40}, low_transfer, False, True)['decision'] == 'reject'

test_capacity(); test_transfer(); test_gate()
print('✅ 后训练系统机制测试通过')

---

🛑 **STOP HERE** 🛑

> 请先完成题目区并运行测试，再查看参考实现。

---

## 参考代码与解析

In [ ]:
def pipeline_capacity(stage_rates, arrival_rate, items_per_work_unit):
    if not stage_rates or any(rate <= 0 for rate in stage_rates.values()):
        raise ValueError('stage rate 必须为正')
    if arrival_rate < 0:
        raise ValueError('arrival_rate 不能为负')
    if set(stage_rates) != set(items_per_work_unit) or any(value <= 0 for value in items_per_work_unit.values()):
        raise ValueError('每个阶段都必须提供正的工作单位倍率')
    effective_rates = {name: float(rate) / items_per_work_unit[name] for name, rate in stage_rates.items()}
    capacity = min(effective_rates.values())
    bottlenecks = sorted(name for name, rate in effective_rates.items() if rate == capacity)
    queue_margins = {name: rate - arrival_rate for name, rate in effective_rates.items()}
    return {'capacity': capacity, 'effective_rates': effective_rates, 'bottlenecks': bottlenecks, 'queue_margins': queue_margins,
            'queue_stable': all(margin >= 0 for margin in queue_margins.values())}

def transfer_ledger(edges, placement):
    rows, total_bytes, fabric_exposed = [], 0.0, {}
    for edge in edges:
        cross_pool = placement[edge['source']] != placement[edge['target']]
        overlap = float(edge.get('overlap_fraction', 0.0))
        bandwidth = float(edge['bandwidth_bytes_per_s'])
        if not 0.0 <= overlap <= 1.0 or bandwidth <= 0:
            raise ValueError('重叠比例须属于 0–1，带宽必须为正')
        items_per_work_unit = float(edge['items_per_work_unit'])
        work_units_per_second = float(edge['work_units_per_second'])
        if items_per_work_unit <= 0 or work_units_per_second < 0:
            raise ValueError('工作单位倍率必须为正，到达率不能为负')
        traffic = float(edge['bytes_per_item']) * items_per_work_unit * work_units_per_second if cross_pool else 0.0
        transfer = traffic / bandwidth
        exposed = transfer * (1.0 - overlap)
        total_bytes += traffic
        if cross_pool:
            fabric_id = edge['fabric_id']
            fabric_exposed[fabric_id] = fabric_exposed.get(fabric_id, 0.0) + exposed
        rows.append(edge | {'cross_pool': cross_pool, 'traffic_bytes_per_s': traffic,
                            'transfer_seconds_per_s': transfer, 'exposed_seconds_per_s': exposed})
    critical_exposed = max(fabric_exposed.values(), default=0.0)
    return {'cross_pool_bytes_per_s': total_bytes, 'fabric_exposed_seconds_per_s': fabric_exposed,
            'critical_exposed_seconds_per_s': critical_exposed, 'rows': rows}

def evaluate_topology(capacity_report, memory_by_pool, memory_limits, transfer_report, version_sync_ok, recovery_ok, max_exposed_s_per_s=0.1):
    over_limit_pools = sorted(pool for pool, used in memory_by_pool.items() if used > memory_limits.get(pool, 0))
    if over_limit_pools:
        return {'decision': 'reject', 'reason': 'memory_limit', 'over_limit_pools': over_limit_pools}
    if not version_sync_ok:
        return {'decision': 'reject', 'reason': 'version_sync'}
    if not recovery_ok:
        return {'decision': 'reject', 'reason': 'recovery_not_ready'}
    if not capacity_report['queue_stable']:
        return {'decision': 'tune', 'reason': 'unstable_queue'}
    if transfer_report['critical_exposed_seconds_per_s'] > max_exposed_s_per_s:
        return {'decision': 'tune', 'reason': 'exposed_transfer'}
    return {'decision': 'accept', 'reason': 'all_gates_passed'}

### 解析

#### TODO 1：计算流水线容量与队列余量

- **机制：** 用每工作单位对象数把各阶段处理率归一到同一工作单位，再由最慢阶段决定流水线容量。
- **不变量：** 到达率必须与阶段容量使用同一单位；任一阶段的等价处理率低于到达率都会造成队列持续增长。
- **测试证据：** 测试覆盖倍率归一、并列瓶颈和稳定/不稳定队列。

#### TODO 2：建立跨池传输账本

- **机制：** 只为跨资源池边建立传输账本，并按共享互连域聚合扣除重叠后的暴露占用。
- **不变量：** 同一互连域的边竞争带宽，需要相加；独立互连域可以并行，关键值取最繁忙互连域而不是全边总和。
- **测试证据：** 测试核对同池边排除、跨池字节数、按域聚合和关键暴露占用。

#### TODO 3：执行拓扑准入

- **机制：** 先执行分池显存、版本确认和恢复能力的硬门槛，再判断队列与关键互连域暴露传输是否需要调优。
- **不变量：** 硬门槛失败必须拒绝；只有硬门槛通过时，容量或传输压力才进入 `tune`。
- **测试证据：** 测试分别覆盖接受、调优和拒绝，并保留触发门槛。

## 相关阅读

- [Part 02 · 85 GRPO 组级对齐项目](./85_GRPO_Groupwise_Alignment_Project.ipynb)
- [Part 02 · 86 在线后训练生命周期与采用决策](./86_DPO_Online_Benchmark.ipynb)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)
- [TRL](https://github.com/huggingface/trl)